# Week 8: Web Scraping (part 2)
## Name: Bria Pickel-Fridman
## 10/19/2026

In [1]:
import csv 
import json # allows us to write JSON files and to interpret requests as JSON
import os # operating system library, let's us do things like making new folders
import requests # allows us to make requests to remote websites, such as those we are scriaping
from time import sleep
from bs4 import BeautifulSoup # the scraping library!


SCRAPED_PAGES_DIRECTORY = "scraped_pages/"
SCRAPED_PAGES_PREFIX = "page"
scraped_pages_count = 0 # keep track of how many pages we have scraped

headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    global scraped_pages_count  # use the scraped_pages_count from the top of the script, instead of making a new local one
    filename = f"{SCRAPED_PAGES_DIRECTORY}{SCRAPED_PAGES_PREFIX}{scraped_pages_count}.html"
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        scraped_pages_count = scraped_pages_count + 1
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)
        scraped_pages_count = scraped_pages_count + 1
    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text


In [9]:
# read data from our URL and interpret it as a "soup"

# specify the URL we want to scrape
url = "https://vilcek.org/art/acoma-polychrome-pot/"
# use that URL to retreive the data from the internet
html = fetch_page(url)
# take the retrieved data and convert into a usable "Soup" object
soup = BeautifulSoup(html, "html.parser")

print(len(html)) # confirms that we have pulled HTML data by showing us how many characters are in the data

141951


In [28]:
title_element = soup.find(class_="side-title").find(name="h1")  # this is the element containing the title
print(title_element.get_text()) # this is the text contained within the title_element
title = title_element.get_text() # declaring a variable

entry_content_element = soup.find(class_="side-title__bottom") # element containing all the paragraphs that mke up the blog post
children = entry_content_element.find_all(True, recursive=False) # grab all the child elements but not the grandchildren


# this is my mpa for figuring out the layout of my div. Uncomment to use
# for index, child in enumerate(children): # it will give me where it exist in the list
#     print(index, child.name, "|", child.get_text())
    

print(children[0].find(name="p").get_text())
artist = children[0].find(name="p").get_text() # declaring a variable
print(children[1].find(name="p").get_text())
date = children[1].find(name="p").get_text() # declaring a variable

entry_paragraphs = []

for child in children:
    entry_paragraphs.append(child.get_text())

print(len(entry_paragraphs))

entry_dictionary = {
    "title": title,
    "paragraphs": entry_paragraphs
}

print(entry_dictionary)

Acoma water jar
Unknown Artist
c. 1920-30
9
{'title': 'Acoma water jar', 'paragraphs': ['\nArtist\nUnknown Artist\n', '\nDate\nc. 1920-30\n', '\nMedium\n\nCeramic\n\n', '\nObject Type\n\nPottery\n\n', '\nDimensions\n10 1/4 x 12 in.  (26 x 30.5 cm)\n', '\nCollecting Area\nNative American Pottery\n', '\nCredit Line\nVilcek Foundation Collection\n', '\nAccession Number\nVF2019.02.03\n', '\nCopyright\n© The Vilcek Foundation\n']}


I am now creating a function that will allow me to repeat the above process for multiple websites. A function is a way o fbundling together processes repeated. 

In [ ]:
def scrape_data(html, url):
    soup = BeautifulSoup(html, "html.parser")
    title_element = soup.find(class_="side-title").find(name="h1")
    title = title_element.get_text()
    entry_content_element = soup.find(class_="side-title__bottom")
    artist = children[0].find(name="p").get_text()
    date = children[1].find(name="p").get_text()
    entry_paragraphs = []

for child in children:
    entry_paragraphs.append(child.get_text())

print(len(entry_paragraphs))

entry_dictionary = {
    "title": title,
    "paragraphs": entry_paragraphs
}

return entry_dictionary

In [ ]:
urls = [
    "https://vilcek.org/art/acoma-polychrome-pot/",
    "https://vilcek.org/art/zuni-polychrome-olla-with-stylized-wing-design/",
    "https://vilcek.org/art/zia-polychrome-pot/",
    "https://vilcek.org/art/san-ildefonso-ceremonial-kiva-pot/",
    "https://vilcek.org/art/zia-four-color-pot/",
    "https://vilcek.org/art/kewa-polychrome-jar/"
    
]

scraped_data = [] # this is an empty list to hold all of the dicitonaries representing the scraped websites

for url in urls: # fir each of the URLs in my list of URLS...
    html=fetch_page(url) # fetch the page, retrieving it from the web and loading it as an html string...
    website_data = scrape_data(html,url) #scrape the data in that html string using scrape data
    scraped_data.append(entry_data) #append that scraped data to the "scraped_data" list

print(scraped_data)

In [ ]:
with open("scraped_data.json", "w", encoding="utf-8") as file: #create a new file. Refer to it as "file"
    json.dump(scraped_data, file, indent=4, ensure_ascii=False

## Homework


In a notebook, list your data sources as properly-formatted markdown URLs. The links should go directly to the datasets, API documentation, or raw data that you intend to use
* [The Met API](https://metmuseum.github.io/)
* [Native Land Digital API](https://api-docs.native-land.ca/)

Outline the steps that you need to take to “wrangle” your dataset so that it is optimized for your particular intervention. This might involve processes of merging, sorting, filtering, cleaning, storing, or formatting your data so that you can accomplish your desired analysis using Python.
** The Met API Data: **
1. Create a list of dictionaries for each of the tribe names and terms I wish to cover in this project.
    - They must be listed as the maker of the object and not an item type by searching `artistOrCulture`= true.
    - ie. object made by or from Pueblo tribe or artist, not an photograph of a Pueblo-style home or Pueblo people.
2. Create another list of dictionaries filtering for objects that meet the criteria from point 1 and contain Geogrpahy information by searching `geoLocation`(string).
    - Specifying where it was made found or sold will be a secondary filter on `graphyType` (string) after I've identified all of the objects with Geography information.
    - I will be looking for `geographyType` strings that contain "Made in", "From", or "Attributed to"
    - Not all objects contain geographic information. I will need to observe if there is a significant gap between the two sets of dictionary.
** Native Land Digital API Data: **
1. Create a map that shows all of the appropriate nameplaces
    - I must consider how to tranliterate some of the nameplaces
    - ie. `ᑲᐚᐦᑲᑐᐢ` as `Kawacatoose`
    - the information about the nameplaces is presented as polygons of coordinates. I think I will need to create overlapping polygons of the information given in the Native Land Digital API and coordinates of the nameplaces provided in th Met API.
    - The Met API does not include coordinates. Geography is presented as string. ie. "Made in Richmond, Virginia, United States"
    - Where will I get them from? Searching each sset individually defeats the purpose. 

Highlight any challenges you anticipate encountering as you wrangle your data. “I need to figure out how to …” or “I know how to x, but I don’t yet know how to y”.
* I need to figure out how to deal with very long lists and pagination. As discovered in my Homework from Week 6, I was unable to successfully get a list of all 660 objects that met my criteria in a list od dictionaries. I think I just need to restructure my work to not use a 'while' loop.
* I need to figure out how to 